# N23 · Multi-turn Chat Template：默认 system 注入 + think token 砍除

> 配套 `labs/l28.5_multiturn_chat_mask/`。
>
> 读完后你应能：
> 1. 复述"单独 tokenize 子串"和"messages 滑窗 delta"两种朴素方案各自掉哪个坑
> 2. 解释 Fixed Base Conversation 为什么能同时绕开两个坑
> 3. 给 multi-turn agentic SFT/RL 数据 pipeline 验证 loss mask 正确性
>
> 参考：[verl tokenization 博客](../github_repo/Awesome-ML-SYS-Tutorial/rlhf/verl/multi-turn/fast_tokenization/multiturn_tokenization_and_masking_ZH.md)


## 1. 朴素方案 1：单独 tokenize 每条 message

看似最直接：把每条 message 单独走 chat template，然后拼接。坑在 chat template 会**条件渲染**：
- 当 message 列表里没有 system 时，模板可能注入默认 system
- 单独 tokenize 一条 user 时，模板会插入默认 system；和原始整体 tokenize 的结果对不齐


In [ ]:
# 用 mock tokenizer 复现 default system 注入坑
import re

class MockTok:
    def __init__(self, default_system_mode=False):
        self.default_system_mode = default_system_mode
    def encode(self, text, add_special_tokens=False):
        return [ord(c) for c in text]
    def apply_chat_template(self, messages, tokenize=False):
        msgs = list(messages)
        if self.default_system_mode and (not msgs or msgs[0]["role"] != "system"):
            msgs = [{"role": "system", "content": "DEFAULT_SYS"}] + msgs
        return "".join(f"<|{m['role']}|>{m['content']}<|/{m['role']}|>" for m in msgs)

tok = MockTok(default_system_mode=True)
user_only = [{"role": "user", "content": "Hello"}]

print("single user message rendered:")
print(" ", repr(tok.apply_chat_template(user_only)))
print("\n注意 DEFAULT_SYS 凭空出现了！")
print("如果你单独 tokenize user 然后拼到上下文，会注入额外不该有的 system tokens。")

## 2. 朴素方案 2：messages 滑窗 delta

比 1 聪明：每次比较 `apply_chat_template(messages[:i+1])` 与 `messages[:i]`，取尾部差异。
但碰到 QwQ/Qwen3 的推理模型会翻车：**当 assistant 不是最后一条时，模板会砍掉 `<think>...</think>`**。

这导致：
- `messages[:i+1]`（assistant 是最后一条 → think 保留）
- `messages[:i+2]`（assistant 不是最后一条 → think 被砍）

两个字符串的尾部差异不是一个干净的 "新增内容"，而是夹杂着前文 think 被删除的负差。


In [ ]:
class MockTokQwq:
    def encode(self, text, add_special_tokens=False):
        return [ord(c) for c in text]
    def apply_chat_template(self, messages, tokenize=False):
        msgs = [dict(m) for m in messages]
        # 砍除非末尾 assistant 的 think
        for i, m in enumerate(msgs):
            if m["role"] == "assistant":
                is_last = all(m2["role"] != "assistant" for m2 in msgs[i+1:])
                if not is_last:
                    msgs[i] = {"role": "assistant",
                               "content": re.sub(r"<think>.*?</think>", "", m["content"])}
        return "".join(f"<|{m['role']}|>{m['content']}<|/{m['role']}|>" for m in msgs)

tok2 = MockTokQwq()
msgs_full = [
    {"role": "system", "content": "S"},
    {"role": "user", "content": "Q1"},
    {"role": "assistant", "content": "<think>SECRET</think>A1"},
    {"role": "user", "content": "Q2"},
]

prev = tok2.apply_chat_template(msgs_full[:3])  # asst is last → think kept
curr = tok2.apply_chat_template(msgs_full[:4])  # asst not last → think dropped
print("prev (asst is last):     ", repr(prev[-50:]))
print("curr (asst not last):    ", repr(curr[-50:]))
print(f"\nlen(prev) = {len(prev)}, len(curr) = {len(curr)}")
print(f"naive delta = curr[len(prev):] = {curr[len(prev):]!r}  ← 这个 delta 是错的！")
print("应该只看到 user Q2 的 tokens，结果包含了 prev 比 curr 多出的 think 片段。")

## 3. Fixed Base Conversation：同时避开两坑

核心洞察：
- 用一个**固定的 base 对话**（system + user）作为前缀
- 每条新 message 单独以 `apply_chat_template(BASE + [msg])` 渲染
- delta = 渲染结果 - base_str

这样：
- BASE 已经包含 system，模板不会注入额外 default system
- 每个新 message 都是 BASE+[msg] 中的最后一条，think 不会被砍


In [ ]:
BASE = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "I am a user."},
]

def tokenize_with_loss_mask(messages, tok):
    base_str = tok.apply_chat_template(BASE)
    ids, loss = [], []
    for m in messages:
        full = tok.apply_chat_template(list(BASE) + [dict(m)])
        delta = full[len(base_str):]
        d_ids = tok.encode(delta)
        ids.extend(d_ids)
        if m["role"] == "assistant":
            loss.extend(d_ids)
        else:
            loss.extend([-100] * len(d_ids))
    return ids, loss

tok3 = MockTokQwq()
msgs = [
    {"role": "system", "content": "S"},
    {"role": "user", "content": "Q1"},
    {"role": "assistant", "content": "<think>SECRET</think>A1"},
    {"role": "user", "content": "Q2"},
    {"role": "assistant", "content": "A2"},
]
ids, loss = tokenize_with_loss_mask(msgs, tok3)
decoded_loss = "".join(chr(t) for t in loss if t != -100)
print("decoded loss-target tokens:")
print(" ", repr(decoded_loss))
print("\n<think>SECRET</think> 完整保留进了 loss。Fixed Base 解决问题。")

## 4. 验证脚本（数据 pipeline 集成）

生产里建议在 SFT 数据生成阶段就跑这个验证：

```python
for sample in sft_dataset:
    ids, loss, attn = tokenize_with_loss_mask(sample.messages, tokenizer)
    
    # 1. 长度对齐
    assert len(ids) == len(loss) == len(attn)
    # 2. 至少一个 assistant token 进 loss
    assert sum(1 for x in loss if x != -100) > 0
    # 3. 没有 user/system 内容串到 loss
    decoded_loss = tokenizer.decode([x for x in loss if x != -100])
    for m in sample.messages:
        if m['role'] in ('user', 'system'):
            assert m['content'] not in decoded_loss
```

## 5. 自检 / 面试题

1. `apply_chat_template(messages, return_assistant_tokens_mask=True)` 为什么不能用作通用方案？（要求 chat template 里显式有 `{% generation %}` 标记，大多数模型不带）
2. 如果 BASE 用空列表 `[]`，会出什么问题？（default system 注入会把 BASE 渲染成包含 default system 的字符串，但 BASE+[user_msg] 也会注入；两者都有 default system 时，`delta` 计算还是对的——所以理论可行，但加 system+user 更稳健，能在更多 chat template 下工作）
3. RL rollout 和 SFT 训练用同一个 chat template，但 deploy 时为什么常常改成另一个？（推理优化：删 think、压缩 system；这造成 train/rollout/deploy 三阶段不一致——verl 默认强制使用训练模板做 rollout）
